# API Models by Benchmark Type

Run selected API model classes against selected benchmark groups. Benchmark grouping lives in `benchmark_groups.py` so other notebooks can reuse it.

## 1. Setup

In [1]:
from pathlib import Path
import importlib
import json
import sys
import traceback

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from benchmark_groups import BENCHMARK_TYPE_TITLES, selected_benchmark_specs
from models.claude_sonnet_4_5 import ClaudeSonnet45
from models.gemini_3_5_flash import Gemini35Flash
from models.gpt_4_1 import GPT41
from models.gpt_4o import GPT4
from models.gpt_5 import GPT5
from models.o4_mini import O4Mini
from models.qwen3_6_plus import Qwen36Plus
from runners.benchmark_run import BenchmarkRun
from runners.execution import run_benchmark_matrix
from runners.model_run import ModelRun

print(f"Repository: {REPO_ROOT}")

C:\Users\Samuel\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repository: C:\Users\Samuel\Documents\GitHub\transformers


## 2. Choose Models and Benchmarks

In [ ]:
RUN_NAME = "api_models"
OUTPUT_DIR = REPO_ROOT / "results" / RUN_NAME
SUMMARY_PATH = OUTPUT_DIR / "run_summary.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_SAMPLES = 20
STREAMING = True
SKIP_EXISTING = False

MODEL_RUNS = [
    ModelRun.from_factory("gpt-4o", GPT4, temperature=0.0),
    # ModelRun.from_factory("gpt-4.1", GPT41, temperature=0.0),
    # ModelRun.from_factory("gpt-5", GPT5, temperature=0.0),
    # ModelRun.from_factory("o4-mini", O4Mini, temperature=0.0),
    # ModelRun.from_factory("claude-sonnet-4-5", ClaudeSonnet45, temperature=0.0),
    # ModelRun.from_factory("gemini-3.5-flash", Gemini35Flash, temperature=0.0),
    # ModelRun.from_factory("qwen3.6-plus", Qwen36Plus, temperature=0.0),
]

SELECTED_BENCHMARKS_BY_TYPE = {
    "A": "ALL",
    "B": "ALL",
    "C": "ALL",
    "E": "ALL",
    "G": "ALL",
    "L": "ALL",
    "P": "ALL",
    "R": "ALL",
    # "A": ["VQA v2.0", "GQA"],
    # "C": ["Flickr30k", "MS COCO Captions"],
}

selected_specs = selected_benchmark_specs(SELECTED_BENCHMARKS_BY_TYPE)
pd.DataFrame(selected_specs)[["type", "type_title", "name", "module", "class"]]

,type,type_title,name,module,class
0,R,Rating,TAD66K,benchmarks.classification.tad66k,TAD66KBenchmark


## 3. Run Matrix

In [3]:
def load_benchmark_class(spec):
    module = importlib.import_module(spec["module"])
    return getattr(module, spec["class"])


summaries = []

for spec in selected_specs:
    benchmark_cls = load_benchmark_class(spec)
    benchmark_name = benchmark_cls.benchmark_name or spec["name"]
    print(f"\n=== Type {spec['type']}: {BENCHMARK_TYPE_TITLES[spec['type']]} / {benchmark_name} ===")

    for model_run in MODEL_RUNS:
        result_path = OUTPUT_DIR / f"{model_run.name}_{benchmark_name}.json"
        if SKIP_EXISTING and result_path.exists():
            saved = json.loads(result_path.read_text(encoding="utf-8"))
            saved_report = saved.get("report", {})
            print(f"[SKIP] {model_run.name} / {benchmark_name}")
            summaries.append({
                "type": spec["type"],
                "type_title": spec["type_title"],
                "model": model_run.name,
                "benchmark": benchmark_name,
                "status": "skipped",
                "num_samples": saved_report.get("num_samples"),
                "results_path": str(result_path),
            })
            continue

        print(f"[RUN] {model_run.name} / {benchmark_name}")
        try:
            benchmark = benchmark_cls(streaming=STREAMING)
            model_run.model.max_new_tokens = benchmark.default_max_new_tokens
            sample_count = min(NUM_SAMPLES, 9) if benchmark.name == "ucf101" else NUM_SAMPLES
            rows = run_benchmark_matrix(
                models=[model_run],
                benchmark_runs=[BenchmarkRun(benchmark=benchmark, num_samples=sample_count)],
                output_dir=OUTPUT_DIR,
            )
            for row in rows:
                summaries.append({
                    "type": spec["type"],
                    "type_title": spec["type_title"],
                    "status": "ok",
                    **row,
                })
            print(f"[OK] {model_run.name} / {benchmark_name}")
        except Exception as exc:
            print(f"[ERROR] {model_run.name} / {benchmark_name}: {type(exc).__name__}: {exc}")
            summaries.append({
                "type": spec["type"],
                "type_title": spec["type_title"],
                "model": model_run.name,
                "benchmark": benchmark_name,
                "status": "error",
                "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc(),
            })

        SUMMARY_PATH.write_text(json.dumps(summaries, ensure_ascii=False, indent=2), encoding="utf-8")

summary_df = pd.DataFrame(summaries)

sample_rows = []
for summary in summaries:
    path = summary.get("results_path")
    if not path:
        continue
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    report = payload.get("report", {})
    for result in report.get("results", []):
        sample_rows.append({
            "type": summary["type"],
            "model": summary["model"],
            "benchmark": summary["benchmark"],
            "sample_index": result.get("index"),
            "total": result.get("total"),
            "correct": result.get("correct"),
            "prediction": result.get("prediction"),
            "error": result.get("error") or result.get("stats", {}).get("error"),
        })

sample_df = pd.DataFrame(sample_rows)
display(summary_df)
sample_df


=== Type R: Rating / tad66k ===
[RUN] gpt-4o / tad66k
[OK] gpt-4o / tad66k


,type,type_title,status,model,benchmark,num_samples,max_new_tokens,results_path
0,R,Rating,ok,gpt-4o,tad66k,20,4,C:\Users\Samuel\Documents\GitHub\transformers\...


,type,model,benchmark,sample_index,total,correct,prediction,error
0,R,gpt-4o,tad66k,1,None,False,8,None
1,R,gpt-4o,tad66k,2,None,False,I'm unable to provide a rating for the aesthet...,None
2,R,gpt-4o,tad66k,3,None,False,I'm unable to provide a rating for the aesthet...,None
3,R,gpt-4o,tad66k,4,None,False,8,None
4,R,gpt-4o,tad66k,5,None,False,7,None
5,R,gpt-4o,tad66k,6,None,True,7,None
6,R,gpt-4o,tad66k,7,None,True,9,None
7,R,gpt-4o,tad66k,8,None,False,9,None
8,R,gpt-4o,tad66k,9,None,False,8,None
9,R,gpt-4o,tad66k,10,None,False,8,None


## 4. Summary by Type

In [4]:
summary_df = pd.DataFrame(summaries)
summary_df.groupby(["type", "type_title", "status"]).size().rename("count").reset_index()

,type,type_title,status,count
0,R,Rating,ok,1
